In [67]:
import os
import sys
sys.path.insert(0, "/home/thomasb/")

import numpy as np
import time
import matplotlib.pyplot as plt
import numba as nb
from scipy.stats import median_abs_deviation
from scipy import linalg
import rfitools
import importlib
from scipy.ndimage import median_filter
import os
from datetime import datetime
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
from astropy.time import Time
import matplotlib.colors as mcolors
from albatros_analysis.src.correlations import timing_solution_class as tsc

import map_utils as mutils

In [68]:
tsobj = tsc.TimingSolution(1753200150, '/scratch/thomasb/timing_solution')

In [69]:
path_satdata = '/scratch/mohanagr/map_output'
npasses = 13

In [70]:
osamp=64
dt = 512 * 4096*osamp/250e6
start_specnum = 1273601
UTC_offset = 1753200128.4140258
UTC_per_spec = 1.638402806904218e-05
unix_start = start_specnum * UTC_per_spec + UTC_offset
ntimes = 160573
fine_tarr = unix_start + (np.arange(ntimes) + 0.5) * dt

chanstart, chanend = 360, 392
freqs = np.arange(chanstart*osamp,chanend*osamp)/osamp * 250e6/4096 # NOT aliased
print(freqs.shape)
print('df', freqs[1]-freqs[0])

(2048,)
df 953.67431640625


In [76]:
for pidx in range(npasses):

    #open npz
    d = np.load(os.path.join(path_satdata, f'jul22_satpass_{pidx}.npz'))

    vis_pass = d['data']
    print('vis_pass shape', vis_pass.shape)
    mask_pass = d['mask']
    print('mask_pass shape', mask_pass.shape)
    unix_pass = d['utc']
    print('utc pass shape', unix_pass.shape)

    print('Unix times around satellite pass', unix_pass.shape)
    print('Duration of satellite pulse (s)', unix_pass[-1]-unix_pass[0])

    # get the clock delays for each baseline at the visibility times
    clock_delays_pass = tsobj.interpolate_delay2(unix_pass, extrapolate=True)
    clock_delays_pass, b = tsobj.all_blines(clock_delays_pass)
    clock_delays_pass *= 1e-9 #turn into ns
    print('clock delays shape', clock_delays_pass.shape)

    #correct for clock
    # vis_pass *= np.exp(
    #     2j*np.pi
    #     *freqs[None, :, None] # (BD, nchans, BD)
    #     *clock_delays_pass.T[:, None, :] # (ntimes, BD, nbl)
    #     )

    vis_pass *= np.exp(
        2j*np.pi
        *freqs[None, None, :] # (BD, nchans, BD)
        *clock_delays_pass.T[:, :, None] # (ntimes, BD, nbl)
        )

    print('pass visibility shape', vis_pass.shape)
    print('pass mask shape', mask_pass.shape)
    print('pass times shape', unix_pass.shape)
    print('pass frequencies shape', vis_pass.shape)
    print('all frequencies shape', freqs.shape)

    path_data = '/scratch/thomasb/mapmaking_dumps/all_sats_science_band'
    os.makedirs(path_data, exist_ok=path_data)
    np.savez(
        os.path.join(path_data, f'satpass_{pidx}.npz'),
        vis = vis_pass,
        mask = mask_pass,
        times = unix_pass,
        freqs = freqs,
    )

vis_pass shape (246, 21, 2048)
mask_pass shape (246, 21, 2048)
utc pass shape (246,)
Unix times around satellite pass (246,)
Duration of satellite pulse (s) 131.53337359428406
number of baselines (containing ref ant) in tau data: 6
number of desired interpolation unix times: 246
number of total data unix times (1560,)
clock delays shape (21, 246)
pass visibility shape (246, 21, 2048)
pass mask shape (246, 21, 2048)
pass times shape (246,)
pass frequencies shape (246, 21, 2048)
all frequencies shape (2048,)
vis_pass shape (245, 21, 2048)
mask_pass shape (245, 21, 2048)
utc pass shape (245,)
Unix times around satellite pass (245,)
Duration of satellite pulse (s) 130.99650263786316
number of baselines (containing ref ant) in tau data: 6
number of desired interpolation unix times: 245
number of total data unix times (1560,)
clock delays shape (21, 245)
pass visibility shape (245, 21, 2048)
pass mask shape (245, 21, 2048)
pass times shape (245,)
pass frequencies shape (245, 21, 2048)
all fr